In [1]:
benchmark_datasets = [
    "Contaminant-detection-in-packaged-cocoa-hazelnut-spread-jars-using-Microwaves-Sensing-and-Machine-Learning-9.5GHz(Urbinati)",
    "MIC",
    "Marketing_Campaign",
    "Wilt",
    "rl",
    "seismic+bumps",
    "Bank_Customer_Churn_Dataset",
    "FICO-HELOC-cleaned",
    "taiwanese_bankruptcy_prediction",
    "telco-customer-churn",
    "adult",
    "naticusdroid+android+permissions+dataset",
    "estimation_of_obesity_levels",
    "led7",
    "maternal_health_risk",
    "wine-quality-white",
    "JapaneseVowels",
    "dry_bean_dataset",
    "letter",
    "walking-activity",
    "Goodreads-Computer-Books",
    "Large-scale_Wave_Energy_Farm_Sydney_100",
    "Mobile_Phone_Market_in_Ghana",
    "stock",
    "BNG(echoMonths)",
    "Bias_correction_r",
    "Large-scale_Wave_Energy_Farm_Sydney_49",
    "sulfur",
    "fried",
    "gas_turbine_CO_and_NOx_emission"
]

In [ ]:
# walking-activity отдельно; остальные группы сбалансированы по оценке времени.

user1 = [
    "walking-activity",
]

user2 = [
    "adult",
    "rl",
    "Mobile_Phone_Market_in_Ghana",
    "Large-scale_Wave_Energy_Farm_Sydney_100",
    "maternal_health_risk",
]

user3 = [
    "fried",
    "dry_bean_dataset",
    "JapaneseVowels",
    "telco-customer-churn",
    "Wilt",
    "Contaminant-detection-in-packaged-cocoa-hazelnut-spread-jars-using-Microwaves-Sensing-and-Machine-Learning-9.5GHz(Urbinati)",
    "Marketing_Campaign",
    "stock",
]

user4 = [
    "gas_turbine_CO_and_NOx_emission",
    "Large-scale_Wave_Energy_Farm_Sydney_49",
    "sulfur",
    "FICO-HELOC-cleaned",
    "taiwanese_bankruptcy_prediction",
    "seismic+bumps",
    "estimation_of_obesity_levels",
]

user5 = [
    "naticusdroid+android+permissions+dataset",
    "letter",
    "BNG(echoMonths)",
    "Bank_Customer_Churn_Dataset",
    "Bias_correction_r",
    "wine-quality-white",
    "led7",
    "MIC",
    "Goodreads-Computer-Books",
]

In [ ]:
import torch

# На сервере укажи индекс выделенной GPU вместо 0.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

config = {
    "datasets": user1,  # На своём аккаунте выбери user1 ... user5.
    "results_dir": "results",
    "gpu_memory_fraction": 0.6,  # Запас относительно серверного лимита 12 ГБ.
    "models": ["modernnca", "catboost"],  # Можно добавить "mlp"
    "seeds": list(range(15)),  # 15 сидов; для каждого новое разбиение 64/16/20
    "tune_seed": 0,
    "n_trials": 100,  # Один подбор на датасет и модель
    "max_epochs": 200,
    "patience": 20,  # Эпох без улучшения для нейросетей
    "max_boost_rounds": 2000,
    "early_stopping_rounds": 50,  # Итераций без улучшения для бустов
    "batch_size": 1024,
    "modernnca": {
        "n_frequencies": 77,
    },
}

if device.type == "cuda":
    torch.cuda.set_device(device)
    torch.cuda.set_per_process_memory_fraction(config["gpu_memory_fraction"], device=device)
print("Device", device)

In [ ]:
from Stream1_DataExtraction import load_dataset
from Stream2_Preprocessing import *


def prepare_run(dataset_name, seed):
    dataset = load_dataset(dataset_name, seed=seed)
    prep = TabularPreprocessor(dataset.numerical_columns, dataset.categorical_columns)
    prep.fit(dataset.X_train)
    train = prep.transform(dataset.X_train, dataset.y_train)
    val = prep.transform(dataset.X_val, dataset.y_val)
    test = prep.transform(dataset.X_test, dataset.y_test)
    return dataset, prep, train, val, test

In [20]:
from Stream3_ModernNCA import *


def build_modern_nca(params, seed, train):
    torch.manual_seed(seed)
    n_num = train.x_num.shape[1] if train.x_num is not None else 0
    n_cat = train.x_cat.shape[1] if train.x_cat is not None else 0
    embedder = PLREmbeddings(n_num, n_frequencies=params["n_frequencies"], d_embedding=params["plr_dim"]) if n_num else None

    return ModernNCAEncoder(
        plr_module=embedder,
        num_continuous=n_num,
        plr_dim=params["plr_dim"],
        num_categorical=n_cat,
        hidden_dim=params["hidden_dim"],
        embed_dim=params["embed_dim"],
        num_layers=params["num_layers"],
        dropout_rate=params["dropout"],
        post_mlp_layers=[],
    )

In [ ]:
from Stream4_Train import *
import optuna


def tune_modern_nca(dataset_run, train_run, val_run):
    train_data = (train_run.x_num, train_run.x_cat, train_run.y)
    val_data = (val_run.x_num, val_run.x_cat, val_run.y)
    nca_params = {
        **config["modernnca"],
        "device": str(device),
        "batch_size": config["batch_size"],
        "patience": config["patience"],
        "verbose": False,
    }

    def nca_objective(trial):
        params = {
            **nca_params,
            "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "dropout": trial.suggest_float("dropout", 0.0, 0.3),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
            "embed_dim": trial.suggest_categorical("embed_dim", [32, 64, 128]),
            "plr_dim": trial.suggest_categorical("plr_dim", [16, 34, 64]),
            "num_layers": trial.suggest_int("num_layers", 1, 2),
            "sample_rate": trial.suggest_float("sample_rate", 0.05, 0.6),
        }
        trial.set_user_attr("config", params)
        trainer_trial = Trainer(build_modern_nca(params, config["tune_seed"], train_run), params)
        trainer_trial.fit(train_data, config["max_epochs"], params["sample_rate"], dataset_run.task_type, dataset_run.n_classes, val_data)
        return trainer_trial.best_score

    study = optuna.create_study(
        direction="minimize" if dataset_run.task_type == "regression" else "maximize",
        sampler=optuna.samplers.TPESampler(seed=config["tune_seed"]),
        storage=optuna_storage,
        study_name=f"{dataset_run.dataset_name}:modernnca:{tuning_tag}",
        load_if_exists=True,
    )
    completed = len(study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,)))
    remaining = max(0, config["n_trials"] - completed)
    print("Optuna", study.study_name, "completed", completed, "remaining", remaining)
    if remaining:
        study.optimize(nca_objective, n_trials=remaining)
    return {**study.best_trial.user_attrs["config"], "device": str(device)}

In [ ]:
from steram5 import compute_metric


def run_modern_nca(seed, dataset_run, train_run, val_run, test_run, params):
    train_data = (train_run.x_num, train_run.x_cat, train_run.y)
    val_data = (val_run.x_num, val_run.x_cat, val_run.y)
    model = build_modern_nca(params, seed, train_run)
    trainer = Trainer(model, params)
    trainer.fit(train_data, config["max_epochs"], params["sample_rate"], dataset_run.task_type, dataset_run.n_classes, val_data)
    print(dataset_run.dataset_name, "modernnca", seed, "best epoch", trainer.best_epoch, "epochs trained", trainer.epochs_trained)

    predictions = trainer.predict(
        train_data,
        (test_run.x_num, test_run.x_cat),
        dataset_run.task_type,
        dataset_run.n_classes,
    )

    if dataset_run.task_type in ("binary", "multiclass"):
        predictions = predictions.argmax(dim=1)

    metric_name, metric_value = compute_metric(
        dataset_run.task_type,
        test_run.y.cpu().numpy(),
        predictions.cpu().numpy(),
    )
    print(dataset_run.dataset_name, "modernnca", seed, metric_name, metric_value)
    return {
        "dataset": dataset_run.dataset_name,
        "model": "modernnca",
        "seed": seed,
        "task_type": dataset_run.task_type,
        "metric": metric_name,
        "metric_value": metric_value,
        "config": params.copy(),
    }

In [ ]:
from steram5 import *
import csv
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

baseline_names = [name for name in config["models"] if name != "modernnca"]
baseline_configs = {
    "catboost": {"iterations": config["max_boost_rounds"], "early_stopping_rounds": config["early_stopping_rounds"]},
    "xgboost": {"n_estimators": config["max_boost_rounds"], "early_stopping_rounds": config["early_stopping_rounds"]},
    "mlp": {"max_epoch": config["max_epochs"], "patience": config["patience"], "batch_size": config["batch_size"]},
}
baseline_configs["mlp"]["device"] = str(device)
baseline_configs["xgboost"].update(device=str(device), tree_method="hist")
if device.type == "cuda":
    baseline_configs["catboost"].update(task_type="GPU", devices=str(device.index or 0), gpu_ram_part=config["gpu_memory_fraction"], boosting_type="Plain")

records = []
result_columns = ["dataset", "model", "seed", "task_type", "metric", "metric_value", "config"]
results_dir = Path(config["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)
optuna_storage = f"sqlite:///{(results_dir / 'optuna.db').resolve().as_posix()}"
tuning_settings = {key: value for key, value in config.items() if key not in ("datasets", "models", "seeds", "n_trials", "results_dir", "gpu_memory_fraction")}
if device.type == "cuda":
    tuning_settings["device"] = device.type  # Подбор CPU и GPU хранится отдельно.
tuning_tag = hashlib.sha256(json.dumps(tuning_settings, sort_keys=True).encode()).hexdigest()[:12]
results_path = results_dir / f"benchmark_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%fZ}.csv"

with results_path.open("x", encoding="utf-8", newline="") as output:
    csv.DictWriter(output, fieldnames=result_columns).writeheader()
print("Results", results_path)


def save_metric(record):
    with results_path.open("a", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=result_columns)
        writer.writerow({**record, "config": json.dumps(record["config"], ensure_ascii=False)})
    records.append(record)

for dataset_name in config["datasets"]:
    print("Dataset", dataset_name)
    dataset_tune, prep_tune, train_tune, val_tune, test_tune = prepare_run(dataset_name, config["tune_seed"])
    best_params = {}

    if "modernnca" in config["models"]:
        best_params["modernnca"] = tune_modern_nca(dataset_tune, train_tune, val_tune)

    for model_name in baseline_names:
        best_params[model_name] = tune_hyperparameters(
            model_name=model_name,
            dataset=dataset_tune,
            seed=config["tune_seed"],
            preprocessor=prep_tune,
            n_trials=config["n_trials"],
            model_config=baseline_configs[model_name],
            storage=optuna_storage,
            study_name=f"{dataset_name}:{model_name}:{tuning_tag}",
        )

    for seed in config["seeds"]:
        dataset_run, prep_run, train_run, val_run, test_run = prepare_run(dataset_name, seed)

        if "modernnca" in config["models"]:
            save_metric(run_modern_nca(seed, dataset_run, train_run, val_run, test_run, best_params["modernnca"]))

        for model_name in baseline_names:
            result = run_experiment(
                model_name=model_name,
                dataset=dataset_run,
                seed=seed,
                config=best_params[model_name],
                preprocessor=prep_run,
            )
            save_metric({
                "dataset": result.dataset,
                "model": result.model,
                "seed": result.seed,
                "task_type": result.task_type,
                "metric": result.metric_name,
                "metric_value": result.metric_value,
                "config": result.config.copy(),
            })
            print(result.dataset, result.model, result.seed, result.metric_name, result.metric_value)

results = pd.DataFrame(records, columns=result_columns)
print(results[["dataset", "model", "seed", "metric", "metric_value"]])